In [70]:
from IPython.display import display, Markdown

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)


In [5]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [6]:
len(documents)

72

In [7]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [8]:
index.search("How does the agentic loop keep calling the model until it stops?")

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [9]:
from rag_helper import RAGBase

In [48]:
class RAG(RAGBase):
    def __init__(self, index, llm_client):
        super().__init__(index, llm_client)

    def search(self, query, num_results=5):

        return self.index.search(
            query,
            num_results=num_results,
        )
    
    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc["content"])

        return "\n".join(lines).strip()
    
    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=input_messages
        )

        return response.choices[0].message.content, response.usage.to_dict()
    
    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer, usage = self.llm(prompt)
        return answer, usage

In [49]:
import os

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

openai_client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ.get("GEMINI_API_KEY")
)

rag = RAG(index=index, llm_client=openai_client)

In [50]:
resp = rag.rag("How does the agentic loop keep calling the model until it stops?")

In [44]:
resp[1]

{'completion_tokens': 243, 'prompt_tokens': 7837, 'total_tokens': 8707}

In [47]:
resp.choices[0].message.content

"Based on the provided context, the agentic loop keeps calling the model until it stops by wrapping the process in a `while` loop (typically a `while True` loop) that monitors whether the model requests any function calls. \n\nHere is how it works step-by-step:\n1. **Tracks Function Calls:** The loop uses a flag (e.g., `has_function_calls = False`) at the start of each iteration.\n2. **Processes Model Output:** When the model responds, the code checks the output. If the response contains a tool/function call, the code runs the tool, appends the tool's output to the message history, and sets `has_function_calls = True`.\n3. **Repeats the Loop:** Because a function call occurred, the loop continues to the next iteration and sends the updated conversation history (including the tool's results) back to the model.\n4. **Stops on Final Answer:** When the model is satisfied and returns a response that contains only a final message with no more function calls, `has_function_calls` remains `Fal

In [52]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [109]:
chunks[:2]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [63]:
index_chunks = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index_chunks.fit(chunks)

In [67]:
rag_chunks = RAG(index=index_chunks, llm_client=openai_client)

In [68]:
resp_chunks = rag.rag("How does the agentic loop keep calling the model until it stops?")

In [72]:
display(Markdown(resp_chunks[0]))

Based on the provided context, the agentic loop keeps calling the model using a `while True` loop that executes until the model returns a response without any function calls. 

Here is how it works step-by-step:
1. **Initialize a flag:** At the start of each iteration, a flag (such as `has_function_calls`) is set to `False`.
2. **Call the model:** The loop calls the model and checks its response.
3. **Execute tools:** If the model returns a `function_call`, the code runs the tool, appends the result to the messages history, and sets the `has_function_calls` flag to `True`.
4. **Check exit condition:** At the end of the iteration, the loop checks the flag. If no function calls were made during that turn (`has_function_calls == False`), the loop breaks (`break`) and stops. Otherwise, it continues to the next iteration.

In [73]:
resp_chunks[1]

{'completion_tokens': 202, 'prompt_tokens': 2489, 'total_tokens': 3277}

In [102]:
def search(query):
    return index.search(
        query,
        num_results=5,
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the course materials for entries matching the given query.",
    "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course materials. Keyword search is used (exact term matching)."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }  
    }

In [87]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [103]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [104]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course materials for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course materials. Keyword search is used (exact term matching).'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [105]:
instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [106]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


In [107]:
result.cost

CostInfo(input_cost=Decimal('0.018636'), output_cost=Decimal('0.002673'), total_cost=Decimal('0.021309'))

In [108]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG"}', call_id='call_2B9TVVG1AwQchEDYmUuGvIAv', name='search', type='function_call', id='fc_0d81d5ff4494c8d3006a37fdf01e5c8192b7c6a30da2c63e3e', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"plain RAG"}', call_id='call_WyPCnVxmOR1ozDRPKxHixma3', name='search', type='function_call', id='fc_0d81d5ff4494c8d3006a37fdf01e708192a5daa6311d2800ea', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop"}', call_id='call_wpLVjrAPwiVUZs5lRcWTy0os', name='search', type='function_call', id='fc_0d81d5ff4